# Spatial Maps — West Africa Precipitation Products
## Comparative Assessment 2001–2020

**Map 1** — Long-term mean annual precipitation (mm/yr) for all 6 products + GPCC reference  
**Map 2** — Spatial bias (product − GPCC) showing systematic over/under-estimation  

**Workflow:**  
1. Export mean annual images from GEE (one per product) → GeoTIFF  
2. Export GPCC mean from your local NetCDF files → GeoTIFF  
3. Plot both maps with cartopy, coordinate ticks, zone boundaries, station markers  

Run the GEE export cells first, download the GeoTIFFs to `MAPS_DIR`, then run the plotting cells.

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────
BASE_DIR  = Path(r'C:\Users\Gebruiker\OneDrive\Spain\Paper 1\precipitation_assessment')
DATA_DIR  = BASE_DIR / 'DATA_DIR'
MAPS_DIR  = BASE_DIR / 'maps'          # GeoTIFFs exported from GEE go here
GPCC_DIR  = DATA_DIR / 'gpcc_raw'      # your existing GPCC .nc files
FIGS_DIR  = BASE_DIR / 'figures'
MAPS_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)

# ── Study area bounding box ──────────────────────────────
LON_MIN, LON_MAX = -18.0, 16.5
LAT_MIN, LAT_MAX =   3.5, 21.5

# ── Products ─────────────────────────────────────────────
PRODUCTS = ['CHIRPS','ERA5_LAND','GPM_IMERG','MERRA2','PERSIANN_CDR','TERRACLIMATE']

PROD_LABELS = {
    'CHIRPS'       : 'CHIRPS\n(0.05°, Satellite-gauge)',
    'ERA5_LAND'    : 'ERA5-Land\n(0.1°, Reanalysis)',
    'GPM_IMERG'    : 'GPM IMERG\n(0.1°, Satellite-gauge)',
    'MERRA2'       : 'MERRA-2\n(~0.5°, Reanalysis)',
    'PERSIANN_CDR' : 'PERSIANN-CDR\n(0.25°, Satellite)',
    'TERRACLIMATE' : 'TerraClimate\n(~0.04°, Interp.)',
}

# ── Station coordinates ───────────────────────────────────
STATIONS = {
    'WA001':(-17.47,14.73,'Dakar'),
    'WA002':( -7.95,12.65,'Bamako'),
    'WA003':( -1.52,12.36,'Ouagadougou'),
    'WA004':(  2.17,13.51,'Niamey'),
    'WA005':(  7.33, 9.07,'Abuja'),
    'WA006':( -0.17, 5.56,'Accra'),
    'WA007':( -3.93, 5.35,'Abidjan'),
    'WA008':(-13.67, 9.53,'Conakry'),
    'WA009':(-13.23, 8.49,'Freetown'),
    'WA010':(-10.80, 6.30,'Monrovia'),
    'WA011':(  1.22, 6.13,'Lomé'),
    'WA012':(  2.42, 6.37,'Cotonou'),
    'WA013':(  8.52,12.05,'Kano'),
    'WA014':( -1.62, 6.69,'Kumasi'),
    'WA015':(-16.68,13.45,'Banjul'),
    'WA016':(-15.97,18.07,'Nouakchott'),
}

print('Config loaded.')
print(f'MAPS_DIR : {MAPS_DIR}')
print(f'GPCC_DIR : {GPCC_DIR}')
print(f'FIGS_DIR : {FIGS_DIR}')


## Step 1 — Export mean annual maps from GEE

Run this cell to submit GEE export tasks.  
Each exports a GeoTIFF of long-term mean mm/day clipped to West Africa.  
Download them from Google Drive to `MAPS_DIR` before running the plotting cells.

In [ ]:
import ee
ee.Initialize(project='ee-desmond')

ASSET_BASE   = 'projects/ee-desmond/assets/'
DRIVE_FOLDER = 'WA_Maps'
START = '2001-01-01'
END   = '2020-12-31'
SCALE = 27830  # 0.25 degrees

ROI = ee.FeatureCollection('projects/ee-desmond/assets/west_africa_boundary0')\
        .geometry().simplify(maxError=5000)

# ── Product collection definitions (matches data_ingestion.py) ──
def get_mean_image(product):
    """Return long-term mean mm/day image for a product."""

    if product == 'CHIRPS':
        ic = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')\
              .filterDate(START, END).select('precipitation')
        return ic.mean().rename('precip_mm_day')

    elif product == 'ERA5_LAND':
        ic = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')\
              .filterDate(START, END).select('total_precipitation_sum')
        def conv(img):
            d   = ee.Date(img.get('system:time_start'))
            dim = d.advance(1,'month').difference(d,'day')
            return img.multiply(1000).divide(dim).rename('precip_mm_day')\
                      .copyProperties(img,['system:time_start'])
        return ic.map(conv).mean().rename('precip_mm_day')

    elif product == 'GPM_IMERG':
        ic = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07')\
              .filterDate(START, END).select('precipitation')
        return ic.mean().multiply(24).rename('precip_mm_day')

    elif product == 'MERRA2':
        # Load from pre-exported yearly climatology assets
        imgs = [ee.Image(f'{ASSET_BASE}climatology_MERRA2_{yr}')\
                  .reduce(ee.Reducer.mean()).rename('precip_mm_day')
                for yr in range(2001, 2021)]
        return ee.ImageCollection(imgs).mean().rename('precip_mm_day')

    elif product == 'PERSIANN_CDR':
        ic = ee.ImageCollection('NOAA/PERSIANN-CDR')\
              .filterDate(START, END).select('precipitation')
        return ic.mean().rename('precip_mm_day')

    elif product == 'TERRACLIMATE':
        ic = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')\
              .filterDate(START, END).select('pr')
        def conv_terra(img):
            d   = ee.Date(img.get('system:time_start'))
            dim = d.advance(1,'month').difference(d,'day')
            return img.divide(dim).rename('precip_mm_day')\
                      .copyProperties(img,['system:time_start'])
        return ic.map(conv_terra).mean().rename('precip_mm_day')


# ── Submit export tasks ────────────────────────────────────
PRODUCTS = ['CHIRPS','ERA5_LAND','GPM_IMERG','MERRA2','PERSIANN_CDR','TERRACLIMATE']
tasks = {}

for prod in PRODUCTS:
    img = get_mean_image(prod).multiply(365.25).clip(ROI)  # mm/day → mm/yr
    task = ee.batch.Export.image.toDrive(
        image          = img.toFloat(),
        description    = f'map_mean_annual_{prod}',
        folder         = DRIVE_FOLDER,
        fileNamePrefix = f'map_mean_annual_{prod}',
        region         = ROI,
        scale          = SCALE,
        crs            = 'EPSG:4326',
        maxPixels      = 1e13,
    )
    task.start()
    tasks[prod] = task
    print(f'  Submitted: map_mean_annual_{prod}.tif')

print(f'\n{len(tasks)} tasks submitted.')
print('Monitor: https://code.earthengine.google.com/tasks')
print(f'Download GeoTIFFs from Drive folder "{DRIVE_FOLDER}" to MAPS_DIR')


## Step 2 — Build GPCC reference GeoTIFF from local files

Uses the GPCC .nc files already downloaded by `download_gpcc.py`.  
Computes mean annual precipitation over 2001–2020 at 1.0° resolution.

In [ ]:
import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path

GPCC_OUT = MAPS_DIR / 'map_mean_annual_GPCC.tif'

if GPCC_OUT.exists():
    print(f'Already exists: {GPCC_OUT.name} — skipping')
else:
    yearly_means = []
    for yr in range(2001, 2021):
        nc = GPCC_DIR / f'full_data_daily_v2022_10_{yr}.nc'
        if not nc.exists():
            print(f'  Missing: {nc.name}')
            continue
        ds = xr.open_dataset(nc)
        # GPCC variable name
        var = 'precip' if 'precip' in ds else list(ds.data_vars)[0]
        # Clip to West Africa bounding box
        da = ds[var].sel(
            lon=slice(LON_MIN, LON_MAX),
            lat=slice(LAT_MAX, LAT_MIN)  # GPCC lat is descending
        )
        # Annual mean mm/day
        yearly_means.append(float(da.mean('time').values.mean()))
        # Store full spatial grid
        if yr == 2001:
            # Get the spatial grid once
            spatial_stack = da.mean('time').values[np.newaxis]  # (1,lat,lon)
            lats = da.lat.values
            lons = da.lon.values
        else:
            spatial_stack = np.concatenate(
                [spatial_stack, da.mean('time').values[np.newaxis]], axis=0
            )
        ds.close()
        print(f'  {yr}: loaded')

    # Mean across all years → mm/day, then × 365.25 → mm/yr
    gpcc_mean_mmyr = np.nanmean(spatial_stack, axis=0) * 365.25
    print(f'\nGPCC mean annual range: {gpcc_mean_mmyr.min():.1f} – {gpcc_mean_mmyr.max():.1f} mm/yr')

    # Write GeoTIFF
    transform = from_bounds(
        lons.min(), lats.min(), lons.max(), lats.max(),
        gpcc_mean_mmyr.shape[1], gpcc_mean_mmyr.shape[0]
    )
    with rasterio.open(
        GPCC_OUT, 'w', driver='GTiff',
        height=gpcc_mean_mmyr.shape[0], width=gpcc_mean_mmyr.shape[1],
        count=1, dtype='float32', crs='EPSG:4326', transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(gpcc_mean_mmyr.astype('float32'), 1)
    print(f'Saved: {GPCC_OUT.name}')


## Step 3 — Load all GeoTIFFs for plotting

In [ ]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

def load_tif(path):
    """Load a GeoTIFF and return (data_2d, extent[W,E,S,N])."""
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    return data, extent


# Load products
rasters = {}
missing = []
for prod in PRODUCTS:
    tif = MAPS_DIR / f'map_mean_annual_{prod}.tif'
    if tif.exists():
        rasters[prod], _ = load_tif(tif)
        print(f'  {prod}: {rasters[prod].shape}  '
              f'range {np.nanmin(rasters[prod]):.0f}–{np.nanmax(rasters[prod]):.0f} mm/yr')
    else:
        missing.append(prod)
        print(f'  MISSING: {tif.name}')

# Load GPCC reference
gpcc_tif = MAPS_DIR / 'map_mean_annual_GPCC.tif'
if gpcc_tif.exists():
    gpcc_data, gpcc_extent = load_tif(gpcc_tif)
    print(f'  GPCC: {gpcc_data.shape}  '
          f'range {np.nanmin(gpcc_data):.0f}–{np.nanmax(gpcc_data):.0f} mm/yr')
else:
    print('GPCC GeoTIFF missing — run Step 2 first')

if missing:
    print(f'\nStill missing: {missing}')
    print('Run Step 1 GEE exports and download the TIFFs before plotting.')
else:
    print('\nAll rasters loaded — ready to plot.')


## Map 1 — Mean annual precipitation (mm/yr)

6-panel layout (2 rows × 3 cols) + GPCC reference panel.  
Coordinate ticks in degrees, ecological zone boundaries, station markers.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import numpy as np
from pathlib import Path

# ── Shared settings ───────────────────────────────────────
PROJ   = ccrs.PlateCarree()
EXTENT = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]
XTICKS = [-15, -10, -5, 0, 5, 10, 15]
YTICKS = [5, 8, 11, 14, 17, 20]

# Colormap and range for annual precip
PRECIP_CMAP = 'YlGnBu'
VMIN, VMAX  = 0, 2800   # mm/yr — adjust if needed

# Load zone boundaries
zone_shp = BASE_DIR / 'ecological_zones_5class' / 'ecological_zones_5class.shp'
zones_gdf = gpd.read_file(zone_shp) if zone_shp.exists() else None
if zones_gdf is None:
    print('Zone shapefile not found — zone boundaries will be skipped')


def add_map_elements(ax, title, show_ylabel=False, show_xlabel=False):
    """Add coastlines, borders, gridlines, ticks, and labels to an axis."""
    ax.set_extent(EXTENT, crs=PROJ)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color='#333333')
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, color='#555555', linestyle=':')
    ax.add_feature(cfeature.OCEAN,     facecolor='#D6EAF8', zorder=0)
    ax.add_feature(cfeature.LAND,      facecolor='none',    zorder=0)

    # Zone boundaries
    if zones_gdf is not None:
        for _, zone in zones_gdf.iterrows():
            ax.add_geometries(
                [zone.geometry], crs=PROJ,
                facecolor='none', edgecolor='#222222',
                linewidth=0.8, linestyle='--'
            )

    # Coordinate gridlines with degree ticks
    gl = ax.gridlines(
        crs=PROJ, draw_labels=True,
        xlocs=XTICKS, ylocs=YTICKS,
        linewidth=0.3, color='gray', alpha=0.5, linestyle='--'
    )
    gl.top_labels    = False
    gl.right_labels  = False
    gl.left_labels   = show_ylabel
    gl.bottom_labels = show_xlabel
    gl.xlabel_style  = {'size': 7, 'color': '#333333'}
    gl.ylabel_style  = {'size': 7, 'color': '#333333'}
    gl.xformatter = mticker.FixedFormatter([f'{abs(x)}°W' if x<0 else f'{x}°E' for x in XTICKS])
    gl.yformatter = mticker.FixedFormatter([f'{y}°N' for y in YTICKS])

    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)


def plot_stations(ax, label_stations=False):
    """Plot gauge station markers."""
    lons = [v[0] for v in STATIONS.values()]
    lats = [v[1] for v in STATIONS.values()]
    ax.scatter(lons, lats, s=18, c='red', marker='^',
               transform=PROJ, zorder=5,
               edgecolors='white', linewidths=0.4,
               label='Gauge stations')
    if label_stations:
        for sid, (lon, lat, name) in STATIONS.items():
            ax.text(lon+0.15, lat+0.15, sid, fontsize=5,
                    transform=PROJ, zorder=6, color='#111111')


# ── Build figure ──────────────────────────────────────────
# Layout: GPCC + 6 products = 7 panels → 2 rows × 4 cols
# Row 0: GPCC (reference), CHIRPS, ERA5_LAND, GPM_IMERG
# Row 1: [empty], MERRA2, PERSIANN_CDR, TERRACLIMATE
panel_order = [
    [('GPCC','GPCC\n(Reference, 1.0°)'), ('CHIRPS', PROD_LABELS['CHIRPS']),
     ('ERA5_LAND', PROD_LABELS['ERA5_LAND']), ('GPM_IMERG', PROD_LABELS['GPM_IMERG'])],
    [(None,None), ('MERRA2', PROD_LABELS['MERRA2']),
     ('PERSIANN_CDR', PROD_LABELS['PERSIANN_CDR']),
     ('TERRACLIMATE', PROD_LABELS['TERRACLIMATE'])],
]

fig = plt.figure(figsize=(18, 9))
fig.patch.set_facecolor('white')

axes = []
for row in range(2):
    for col in range(4):
        ax = fig.add_subplot(
            2, 4, row*4 + col + 1,
            projection=PROJ
        )
        axes.append((row, col, ax))

img_mappable = None

for row_idx, row_panels in enumerate(panel_order):
    for col_idx, (prod_key, title) in enumerate(row_panels):
        ax_idx = next(ax for r,c,ax in axes if r==row_idx and c==col_idx)

        if prod_key is None:
            ax_idx.set_visible(False)
            continue

        show_y = (col_idx == 0)
        show_x = (row_idx == 1)
        add_map_elements(ax_idx, title, show_y, show_x)

        # Load raster
        if prod_key == 'GPCC':
            data = gpcc_data
            extent_plot = gpcc_extent
        else:
            tif = MAPS_DIR / f'map_mean_annual_{prod_key}.tif'
            if not tif.exists():
                ax_idx.text(0.5, 0.5, f'{prod_key}\nNot yet downloaded',
                            ha='center', va='center', transform=ax_idx.transAxes,
                            fontsize=8, color='gray')
                continue
            data, extent_plot = load_tif(tif)

        im = ax_idx.imshow(
            data, origin='upper',
            extent=extent_plot,
            transform=PROJ,
            cmap=PRECIP_CMAP,
            vmin=VMIN, vmax=VMAX,
            interpolation='bilinear',
            zorder=1
        )
        if img_mappable is None:
            img_mappable = im

        plot_stations(ax_idx, label_stations=(prod_key=='GPCC'))

        # Panel letter label
        label = chr(65 + row_idx*4 + col_idx)  # A, B, C...
        ax_idx.text(0.02, 0.97, f'({label})', transform=ax_idx.transAxes,
                    fontsize=9, fontweight='bold', va='top',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

# Shared colorbar
cbar_ax = fig.add_axes([0.35, 0.04, 0.30, 0.018])
cbar = fig.colorbar(img_mappable, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Mean annual precipitation (mm/yr)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Legend for stations
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='^', color='w', markerfacecolor='red',
           markersize=6, label='Gauge stations (n=16)'),
    Line2D([0],[0], color='#222222', lw=0.8, linestyle='--',
           label='Ecological zone boundary'),
]
fig.legend(handles=legend_elements, loc='lower left',
           bbox_to_anchor=(0.02, 0.02), fontsize=8, framealpha=0.9)

fig.suptitle(
    'Mean Annual Precipitation (2001–2020) — West Africa\n'
    'Six global products vs GPCC gauge-based reference',
    fontsize=12, fontweight='bold', y=0.98
)

fig.tight_layout(rect=[0, 0.07, 1, 0.96])
out = FIGS_DIR / 'map01_mean_annual_precip.png'
fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {out.name}')
plt.show()


## Map 2 — Spatial bias relative to GPCC (mm/yr)

5-panel layout (one per satellite/reanalysis product).  
Diverging colormap: red = product wetter, blue = drier than GPCC.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np

# Bias colormap settings
BIAS_CMAP = 'RdBu_r'  # red = positive (product wetter), blue = negative (drier)
BIAS_MAX  = 600       # mm/yr — adjust based on your data range

bias_norm = TwoSlopeNorm(vmin=-BIAS_MAX, vcenter=0, vmax=BIAS_MAX)

# Layout: 2 rows × 3 cols, last panel = empty or summary stats
# Products for bias: all 6 vs GPCC
bias_order = [
    [('CHIRPS', PROD_LABELS['CHIRPS']),
     ('ERA5_LAND', PROD_LABELS['ERA5_LAND']),
     ('GPM_IMERG', PROD_LABELS['GPM_IMERG'])],
    [('MERRA2', PROD_LABELS['MERRA2']),
     ('PERSIANN_CDR', PROD_LABELS['PERSIANN_CDR']),
     ('TERRACLIMATE', PROD_LABELS['TERRACLIMATE'])],
]

fig2, axs2 = plt.subplots(
    2, 3, figsize=(15, 9),
    subplot_kw={'projection': PROJ}
)
fig2.patch.set_facecolor('white')

bias_mappable = None

for row_idx, row_panels in enumerate(bias_order):
    for col_idx, (prod_key, title) in enumerate(row_panels):
        ax = axs2[row_idx][col_idx]
        show_y = (col_idx == 0)
        show_x = (row_idx == 1)
        add_map_elements(ax, title, show_y, show_x)

        tif = MAPS_DIR / f'map_mean_annual_{prod_key}.tif'
        if not tif.exists():
            ax.text(0.5, 0.5, f'{prod_key}\nNot yet downloaded',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=8, color='gray')
            continue

        prod_data, prod_extent = load_tif(tif)

        # Regrid GPCC to product resolution for subtraction
        # Simple approach: resize GPCC to match product array
        from scipy.ndimage import zoom
        if gpcc_data.shape != prod_data.shape:
            zy = prod_data.shape[0] / gpcc_data.shape[0]
            zx = prod_data.shape[1] / gpcc_data.shape[1]
            gpcc_resized = zoom(gpcc_data, (zy, zx), order=1, prefilter=False)
        else:
            gpcc_resized = gpcc_data

        bias = prod_data - gpcc_resized
        print(f'{prod_key}: bias range {np.nanmin(bias):.0f} to {np.nanmax(bias):.0f} mm/yr'
              f'  |  mean bias = {np.nanmean(bias):.1f} mm/yr')

        im = ax.imshow(
            bias, origin='upper',
            extent=prod_extent,
            transform=PROJ,
            cmap=BIAS_CMAP,
            norm=bias_norm,
            interpolation='bilinear',
            zorder=1
        )
        if bias_mappable is None:
            bias_mappable = im

        plot_stations(ax)

        # PBIAS annotation from validation data
        import pandas as pd
        vbz = pd.read_csv(DATA_DIR / 'validation_by_zone.csv')
        # West Africa pooled PBIAS from validation_overall
        vo  = pd.read_csv(DATA_DIR / 'validation_overall.csv')
        pbias_val = vo[vo['product']==prod_key]['pbias'].values
        if len(pbias_val):
            ax.text(0.98, 0.03,
                    f'PBIAS = {pbias_val[0]:+.1f}%',
                    transform=ax.transAxes,
                    fontsize=8, ha='right', va='bottom',
                    bbox=dict(facecolor='white', alpha=0.8,
                              edgecolor='#cccccc', pad=2))

        label = chr(65 + row_idx*3 + col_idx)
        ax.text(0.02, 0.97, f'({label})', transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top',
                bbox=dict(facecolor='white', alpha=0.7,
                          edgecolor='none', pad=2))

# Shared colorbar
cbar2_ax = fig2.add_axes([0.25, 0.04, 0.50, 0.018])
cbar2 = fig2.colorbar(bias_mappable, cax=cbar2_ax, orientation='horizontal')
cbar2.set_label('Bias vs GPCC reference (mm/yr)  |  Red = product wetter, Blue = product drier',
                fontsize=9)
cbar2.ax.tick_params(labelsize=8)

fig2.legend(handles=legend_elements, loc='lower left',
            bbox_to_anchor=(0.01, 0.07), fontsize=8, framealpha=0.9)

fig2.suptitle(
    'Spatial Bias Relative to GPCC (2001–2020) — West Africa\n'
    'Product mean annual precipitation minus GPCC reference (mm/yr)',
    fontsize=12, fontweight='bold', y=0.98
)

fig2.tight_layout(rect=[0, 0.07, 1, 0.96])
out2 = FIGS_DIR / 'map02_spatial_bias_vs_gpcc.png'
fig2.savefig(out2, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {out2.name}')
plt.show()


## Optional Map 3 — Inter-product standard deviation (mm/yr)

Shows pixel-wise disagreement across all 6 products.  
High std dev = where ensemble approach is most needed.

In [ ]:
# Load all product rasters at a common resolution
# Use CHIRPS grid as reference (finest resolution)
from scipy.ndimage import zoom

ref_tif = MAPS_DIR / 'map_mean_annual_CHIRPS.tif'
if not ref_tif.exists():
    print('CHIRPS GeoTIFF not found — skipping Map 3')
else:
    ref_data, ref_extent = load_tif(ref_tif)
    target_shape = ref_data.shape

    stacked = [ref_data]
    for prod in ['ERA5_LAND','GPM_IMERG','MERRA2','PERSIANN_CDR','TERRACLIMATE']:
        tif = MAPS_DIR / f'map_mean_annual_{prod}.tif'
        if not tif.exists():
            print(f'  Missing {prod} — skipping')
            continue
        data, _ = load_tif(tif)
        if data.shape != target_shape:
            zy = target_shape[0] / data.shape[0]
            zx = target_shape[1] / data.shape[1]
            data = zoom(data, (zy, zx), order=1, prefilter=False)
        stacked.append(data)

    std_map = np.nanstd(np.stack(stacked, axis=0), axis=0)
    print(f'Inter-product std dev: mean={np.nanmean(std_map):.0f} '
          f'max={np.nanmax(std_map):.0f} mm/yr')

    fig3, ax3 = plt.subplots(1, 1, figsize=(10, 7),
                              subplot_kw={'projection': PROJ})
    fig3.patch.set_facecolor('white')

    add_map_elements(ax3,
        'Inter-product standard deviation (mm/yr)\nHigher = greater uncertainty across products',
        show_ylabel=True, show_xlabel=True)

    im3 = ax3.imshow(
        std_map, origin='upper', extent=ref_extent,
        transform=PROJ, cmap='OrRd',
        vmin=0, vmax=np.nanpercentile(std_map, 95),
        interpolation='bilinear', zorder=1
    )
    plot_stations(ax3, label_stations=True)

    cbar3 = fig3.colorbar(im3, ax=ax3, orientation='vertical',
                          fraction=0.03, pad=0.04, shrink=0.8)
    cbar3.set_label('Std dev across 6 products (mm/yr)', fontsize=9)

    fig3.suptitle(
        'Inter-product Agreement — West Africa 2001–2020\n'
        'Pixel-wise standard deviation across CHIRPS, ERA5-Land, GPM-IMERG,\n'
        'MERRA-2, PERSIANN-CDR, TerraClimate',
        fontsize=11, fontweight='bold'
    )
    fig3.tight_layout()
    out3 = FIGS_DIR / 'map03_interproduct_std.png'
    fig3.savefig(out3, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out3.name}')
    plt.show()


## Requirements

```bash
pip install cartopy rasterio scipy geopandas xarray matplotlib earthengine-api
```

**Cartopy on Windows**: if `pip install cartopy` fails, use conda:
```bash
conda install -c conda-forge cartopy -n precip
```

**GeoTIFF download order**:
1. Run the GEE export cell
2. Wait for all 6 tasks to complete (~5–15 min each)
3. Download from Google Drive folder `WA_Maps` to your `MAPS_DIR`
4. Run Step 2 (GPCC from local .nc files)
5. Run Steps 3–4 (load + plot)

**Tweak parameters**:
- `VMIN / VMAX`: colorbar range for Map 1 (mm/yr)
- `BIAS_MAX`: symmetric colorbar range for Map 2
- `XTICKS / YTICKS`: coordinate tick positions in degrees
- `label_stations=True` on any panel to show station IDs
- `SCALE`: GEE export resolution in metres (27830 ≈ 0.25°)